In [3]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

# ※ naver open API를 활용하여 네이버지식인 "전주여행"과 "경주여행"을 검색 -> 명사만 추출(re ? ) -> 빈도분석(DataFrame) -> 빈도 시각화(워드클라우드, Text) -> Word2Vec

# 1. 네이버 open API를 활용하여 검색 추출
- 검색어, no, title, link, description, title + ' ' + description

In [4]:
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path='env.txt')
# print(os.getenv('Client_ID'))
# print(os.getenv('Client_Secret'))

True

In [45]:
import requests
import pandas as pd
import json
from dotenv import load_dotenv
import os
load_dotenv()
# 환경변수에서 API키 읽기
client_id = os.getenv('Client_ID')
client_secret = os.getenv('Client_Secret')
keyword = '전주여행'
cnt = 10
# API 요청 URL 정의
url = 'https://openapi.naver.com/v1/search/kin.json?query={}&display={}'.format(keyword, cnt)
# HTTP 요청 헤더 설정
headers = {
    'X-Naver-Client-Id': client_id,
    'X-Naver-Client-Secret': client_secret
}
# API 요청
response = requests.get(url, headers=headers)
# response.status_code 200 확인
items = response.json()['items']
items_list = []
for idx, item in enumerate(items):
    title = item.get('title').replace('<b>', '').replace('</b>', '')
    link = item.get('link')
    desc = item.get('description').replace('<b>', '').replace('</b>', '')
    article = title + ' ' + desc
    items_list.append({
        '검색어':keyword,
        'no':idx+1,
        'title':title,
        'link':link,
        'description':desc,
        'article':article
    })
# items_list

In [60]:
# 함수 작성
def get_naver_in(keyword, cnt):
    import requests
    import pandas as pd
    import json
    from dotenv import load_dotenv
    import os
    load_dotenv()
    client_id = os.getenv('Client_ID')
    client_secret = os.getenv('Client_Secret')
    url = 'https://openapi.naver.com/v1/search/kin.json?query={}&display={}'.format(keyword, cnt)
    headers = {
        'X-Naver-Client-Id': client_id,
        'X-Naver-Client-Secret': client_secret
    }
    response = requests.get(url, headers=headers)
    items = response.json()['items']
    items_list = []
    for idx, item in enumerate(items):
        title = item.get('title').replace('<b>', '').replace('</b>', '')
        link = item.get('link')
        desc = item.get('description').replace('<b>', '').replace('</b>', '')
        # article = title + ' ' + desc
        items_list.append({
            '검색어':keyword,
            'no':idx+1,
            'title':title,
            'link':link,
            'description':desc})
    return pd.DataFrame(items_list)

In [26]:
# if(rescode==200):
#     response_body = response.read() # 바이트(bytes)형태로 읽어옴 -> 문자열로 변환해야함
#     print(type(response_body.decode('utf-8'))) # 바이트 데이터를 utf-8로 디코딩 후 문자열 변환
#     import json
#     data = json.loads(response_body.decode('utf-8')) # json형태의 str을 딕셔너리로
#     print(type(data))
# else:
#     print("Error Code:" + rescode)

In [17]:
# response = urllib.request.urlopen(request)

# print("상태 코드:", response.getcode())
# print("응답 URL:", response.geturl())
# print("응답 헤더:")
# print(response.info())
# print("본문:")
# print(response.read().decode('utf-8'))

In [21]:
# API 응답 결과는 보통 items라는 키 안에 검색 결과 목록이 리스트 형태로 들어있음
# 이 리스트를 먼저 가져와야 함
# items = data.get('items', [])
# items

# 2. 명사만 추출(re ? )

In [64]:
jeonju_df = get_naver_in('전주여행', 10)
gyeongju_df = get_naver_in('경주여행', 10)
jeonju_df.head(1)

,검색어,no,title,link,description
0,전주여행,1,전주 가볼만한곳 추천 받아요,https://kin.naver.com/qna/detail.naver?d1id=9&...,... 추억의 7080 #다양한체험 #7080감성 #추억여행 #테마박물관 #유익한시...


In [70]:
from konlpy.tag import Okt
import re
okt = Okt()
# def clean(text):
#     return re.sub(r'[“”=+,#/\?:^$.@*\"※~&%ㆍ!』\\‘|\(\)\[\]\<\>`\'…》]', text)
def clean(text):
    if isinstance(text, str):  # 문자열인 경우에만 처리
        return re.sub(r'<.*?>', '', text)
    return ""  # 문자열이 아니면 빈 문자열 반환
def extract_nouns(text):
    if isinstance(text, str):  # 문자열일 때만 처리
        cleaned = clean(text)
        return okt.nouns(cleaned)
    else:
        return []  # 문자열이 아니면 빈 리스트

# 적용
jeonju_df['description_nouns'] = jeonju_df['description'].apply(extract_nouns)

In [71]:
jeonju_df

,검색어,no,title,link,description,description_nouns
0,전주여행,1,전주 가볼만한곳 추천 받아요,https://kin.naver.com/qna/detail.naver?d1id=9&...,... 추억의 7080 #다양한체험 #7080감성 #추억여행 #테마박물관 #유익한시...,"[추억, 전북, 전북, 투어, 패스, 통합, 이용권, 날]"
1,전주여행,2,전주여행갈려고하는데요!,https://kin.naver.com/qna/detail.naver?d1id=9&...,전주여행을 갈려고하는데요!아는사람과 갈려고하는데 호텔은 좋은가격에 정했고~ 음..2...,"[여행, 사람과, 호텔, 가격, 정, 음, 박, 얼마, 정도, 맛집, 카페, 추천,..."
2,전주여행,3,전주 1박 2일 여행 관련 질문~,https://kin.naver.com/qna/detail.naver?d1id=9&...,... 전주 여행을 계획하고 계시다니 정말 좋은 선택을 하셨네요. 전주는 한옥마을과...,"[전주, 여행, 계획, 정말, 선택, 전주, 옥, 마을, 자연, 경관, 음식, 동네..."
3,전주여행,4,중2 여학생 친구와 전주여행,https://kin.naver.com/qna/detail.naver?d1id=9&...,안녕하세요!중 2(15살)여학생입니다.친한친구와 둘이서 전주 여행을 가기로 했는데 ...,"[중, 살, 여학생, 친한친구, 둘이서, 전주, 여행, 가기, 어디, 가야, 저희,..."
4,전주여행,5,부모닝 모시고 전주여행,https://kin.naver.com/qna/detail.naver?d1id=9&...,"부모님 모시고 전주여행가려는데 명소, 식당 추천부탁드립니다 -------------...","[부모님, 모시, 여행, 명소, 식당, 추천, 전주, 옥, 마을, 완산구, 덕진구,..."
5,전주여행,6,2박3일 전주여행,https://kin.naver.com/qna/detail.naver?d1id=9&...,이번주 토~월 2박3일 여자친구와 전주여행을 가는데 여행경로 좀 알려주세요~~ 토요...,"[이번, 주, 토, 월, 박, 여자친구, 여행, 여행, 경로, 좀, 토요일, 비, ..."
6,전주여행,7,전주 1박2일 여행,https://kin.naver.com/qna/detail.naver?d1id=12...,"... 저녁엔 전주천 산책길 따라 걷거나, 한옥마을 근처에서 야경 즐기시는 것도 너...","[저녁, 전주천, 산책길, 옥, 마을, 근처, 야경, 것, 혹시, 일정, 관심, 테..."
7,전주여행,8,전주 여행코스 추천이요~!,https://kin.naver.com/qna/detail.naver?d1id=9&...,영화제 기간 일정으로 2박3일 전주여행 좀 가볼까 해서요. 여행코스 추천 좀 해주세...,"[영화제, 기간, 일정, 박, 전주, 여행, 좀, 여행, 코스, 추천, 좀, 어차피..."
8,전주여행,9,전주 여행,https://kin.naver.com/qna/detail.naver?d1id=9&...,전주로 커플여행 가려고 하는데 추천 좀 해주세요 숙소는 잡았고 놀거리랑 맛집 등등 ...,"[전주, 커플, 여행, 추천, 좀, 숙소, 놀, 거리, 맛집, 등등, 추천, 해주시..."
9,전주여행,10,추석 전주여행,https://kin.naver.com/qna/detail.naver?d1id=9&...,16~18일 전주로 여행 가려고 하는데 그때 한옥마을같은 관광지들은 문 닫을까요.....,"[전주, 여행, 그때, 옥, 마을, 관광지, 문, 추석, 당일, 문, 전주, 이벤트..."


# 3. 빈도분석(DataFrame)

# 4. 빈도 시각화(워드클라우드, Text)

# 5. Word2Vec